# Hierarchy loss in training: does `lifted ⊂ chip` actually get learned?

*Tutorial for issue #67 — nothing here is exported to `src/` (no `#| export`, not run by `nbdev-test`). CPU-only, synthetic data.*

Some classes must sit **inside** another class: a `lifted` pin is only real where there is a `chip`.
The library has had the pieces for a while (a loss term, config parsing, inference clipping) but `train()`
never switched the loss term on. Since #67 it does, controlled by two config keys:

```yaml
train:
  hierarchy:
    lifted_subset_of_chip: true   # which classes nest
    weight: 0.5                   # how hard to push; 0 / missing = off
```

This notebook checks three things, in order:

1. **Wiring** — the loss really rises when `lifted` leaks outside `chip`, and a weight of `0` changes nothing.
2. **Training A/B** — run the real `train()` twice (weight `0` vs `>0`), same data and seed, and compare how often `lifted` leaks, on the *raw* predictions.
3. **Reading the result honestly** — one run is not proof; what to look at before trusting a number.

In [1]:
# setup — all imports live here; CPU-only, seeded, synthetic data throughout
import tempfile
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from segmentation_tools_pt.losses import SegLoss, resolve_loss_weights
from segmentation_tools_pt.train import train
from segmentation_tools_pt.metrics import subset_violation_rate
from segmentation_tools_pt.inference import apply_hierarchy

CHIP, LIFTED = 0, 1
PAIRS = [(LIFTED, CHIP)]          # (sub_idx, super_idx) — what resolve_hierarchy_pairs() returns from the YAML
torch.manual_seed(0)

## 1. Wiring check on the loss itself

Build logits where `lifted` fires everywhere, including far outside the `chip` box. A loss that knows about the
hierarchy must charge extra for that; a loss with weight `0` (or no pairs) must give **exactly** the old number.

In [2]:
tg = torch.zeros(1, 2, 8, 8); tg[0, CHIP, 2:6, 2:6] = 1; tg[0, LIFTED, 3:5, 3:5] = 1     # lifted strictly inside chip
lg = torch.full((1, 2, 8, 8), -4.0); lg[0, CHIP, 2:6, 2:6] = 4.0; lg[0, LIFTED] = 4.0       # lifted leaks over the whole image
mk = lambda **kw: SegLoss(w_tversky=0.5, w_dice=0.5, task='multilabel', **kw)

plain = mk()(lg, tg)
zero_w = mk(w_hierarchy=0.0, hierarchy_pairs=PAIRS)(lg, tg)
with_w = mk(w_hierarchy=1.0, hierarchy_pairs=PAIRS)(lg, tg)
print(f"plain loss        : {float(plain['loss']):.4f}")
print(f"weight 0          : {float(zero_w['loss']):.4f}   (must equal plain)")
print(f"weight 1          : {float(with_w['loss']):.4f}   hierarchy term = {float(with_w['loss_hierarchy']):.4f}")
assert torch.equal(plain['loss'], zero_w['loss'])
assert float(with_w['loss']) > float(plain['loss'])

plain loss        : 0.4503
weight 0          : 0.4503   (must equal plain)
weight 1          : 1.1733   hierarchy term = 0.7230


The weight can also differ per loss phase. A phase without its own `w_hierarchy` falls back to the run-wide
`train.hierarchy.weight`; `resolve_loss_weights` now returns it as the 5th entry and `SegLoss.set_weights(*weights)` takes it directly.

In [3]:
phases = [
    {'until_epoch': 8,  'w_tversky': 0.5, 'w_dice': 0.5, 'w_boundary': 0.0, 'w_hierarchy': 0.0},   # warm-up: off
    {'until_epoch': -1, 'w_tversky': 0.5, 'w_dice': 0.5, 'w_boundary': 0.0},                       # inherits the run-wide weight
]
for ep in (0, 7, 8, 20):
    print(f'epoch {ep:2d} ->', resolve_loss_weights(ep, phases, 0.5))
assert resolve_loss_weights(0, phases, 0.5)[4] == 0.0 and resolve_loss_weights(8, phases, 0.5)[4] == 0.5

epoch  0 -> (0.5, 0.5, 0.0, 0.0, 0.0)
epoch  7 -> (0.5, 0.5, 0.0, 0.0, 0.0)
epoch  8 -> (0.5, 0.5, 0.0, 0.0, 0.5)
epoch 20 -> (0.5, 0.5, 0.0, 0.0, 0.5)


Guard rails fail **before** training starts, with the fix in the message:

In [4]:
for kw in [dict(task='multiclass', num_classes=2, hierarchy_pairs=PAIRS), dict(task='multilabel', num_classes=2, hierarchy_pairs=None)]:
    try:
        train(root=None, model=nn.Conv2d(3, kw['num_classes'], 1), train_loader=[], val_loader=[], num_epochs=1,
              device_str='cpu', checkpoint_dir=tempfile.mkdtemp(), use_amp=False, resume=False, w_hierarchy=0.5, **kw)
    except ValueError as e:
        print('ValueError:', e, '\n')

ValueError: w_hierarchy > 0 needs task='multilabel' (overlapping sigmoid channels), got task='multiclass'. Set train.task: multilabel or train.hierarchy.weight: 0. 

ValueError: w_hierarchy > 0 but no hierarchy pairs were resolved. Add e.g. `train.hierarchy.lifted_subset_of_chip: true` and make sure both names are in data.class_names. 



## 2. A/B with the real `train()`

**Synthetic task.** Each 32×32 image has a `chip` square and a `lifted` blob that is always inside it. A second blob that
*looks* like `lifted` (the *decoy*) sits outside the chip and is labelled `0`: a small model is tempted to light up `lifted` there.
That temptation is exactly what the hierarchy loss is meant to remove.

In [5]:
H = W = 32
class Syn(Dataset):
    def __init__(self, n, seed): self.n, self.seed = n, seed
    def __len__(self): return self.n
    def __getitem__(self, i):
        g = np.random.default_rng(self.seed * 10_000 + i)
        chip = np.zeros((H, W), np.float32); y0, x0 = g.integers(4, 12, 2); chip[y0:y0 + 16, x0:x0 + 16] = 1
        lift = np.zeros((H, W), np.float32); ly, lx = g.integers(y0, y0 + 10), g.integers(x0, x0 + 10)
        lift[ly:ly + 6, lx:lx + 6] = 1; lift *= chip                                   # ground truth obeys lifted ⊂ chip
        decoy = np.zeros((H, W), np.float32); dy, dx = g.integers(0, 26, 2); decoy[dy:dy + 6, dx:dx + 6] = 1; decoy *= (1 - chip)
        img = np.stack([chip * .6 + g.normal(0, .3, (H, W)),
                        (lift + decoy) * .8 + g.normal(0, .5, (H, W)),                 # lifted evidence, decoys look the same
                        g.normal(0, .3, (H, W))]).astype(np.float32)
        return {'image': torch.from_numpy(img), 'mask': torch.from_numpy(np.stack([chip, lift])), 'sample_id': str(i)}

def tiny_net(): return nn.Sequential(nn.Conv2d(3, 4, 3, padding=1), nn.ReLU(), nn.Conv2d(4, 2, 1))   # deliberately small
val_loader = DataLoader(Syn(32, 1), batch_size=8)

`run_one` trains on the same frozen data and seed, differing only in the hierarchy settings, then measures the **raw** predictions
(before `apply_hierarchy`): the share of predicted `lifted` pixels that lie outside predicted `chip` (`subset_violation_rate`, per image, averaged),
the `lifted` IoU, and what the inference-time clip would have changed.

In [6]:
def iou(a, b):
    a, b = a.bool(), b.bool(); u = (a | b).sum()
    return float((a & b).sum() / u) if u else 1.0

def run_one(seed, w=0.0, phases=None, epochs=25):
    torch.manual_seed(seed)
    train_loader = DataLoader(Syn(64, 0), batch_size=8, shuffle=True, generator=torch.Generator().manual_seed(seed))
    model = tiny_net()
    with tempfile.TemporaryDirectory() as d:
        hist = train(root=None, model=model, train_loader=train_loader, val_loader=val_loader, num_classes=2, num_epochs=epochs,
                     batch_size=8, encoder_lr=3e-3, decoder_lr=3e-3, device_str='cpu', checkpoint_dir=d, use_amp=False,
                     resume=False, task='multilabel', hierarchy_pairs=PAIRS, w_hierarchy=w, loss_phases=phases, seed=seed, num_workers=0)
    model.eval(); viol, iou_raw, iou_clip, gt = [], [], [], []
    with torch.no_grad():
        for b in val_loader:
            raw = torch.sigmoid(model(b['image'])) > 0.5
            clipped = apply_hierarchy(raw, PAIRS)
            for k in range(len(raw)):
                viol.append(subset_violation_rate(raw[k, LIFTED].numpy(), raw[k, CHIP].numpy()))
                iou_raw.append(iou(raw[k, LIFTED], b['mask'][k, LIFTED])); iou_clip.append(iou(clipped[k, LIFTED], b['mask'][k, LIFTED]))
    return dict(violation=np.mean(viol), lifted_iou_raw=np.mean(iou_raw), lifted_iou_clipped=np.mean(iou_clip),
                chip_iou=hist[-1]['val_iou_class0'], logged_term=hist[-1].get('train_loss_hierarchy'))

In [7]:
import contextlib, io
warmup = [{'until_epoch': 8, 'w_tversky': .5, 'w_dice': .5, 'w_boundary': 0., 'w_hierarchy': 0.},
          {'until_epoch': -1, 'w_tversky': .5, 'w_dice': .5, 'w_boundary': 0.}]
arms = {'A: weight 0': dict(w=0.0), 'B: weight 0.1': dict(w=0.1), 'B: weight 0.5': dict(w=0.5), 'B: 0 → 0.5 after epoch 8': dict(w=0.5, phases=warmup)}
rows = []
for seed in (0, 1, 2):
    for name, kw in arms.items():
        with contextlib.redirect_stdout(io.StringIO()):      # silence the per-epoch log
            r = run_one(seed, **kw)
        rows.append(dict(seed=seed, arm=name, **r))
def table(rows, cols):
    print(' | '.join(f'{c:>24}' if i == 0 else f'{c:>18}' for i, c in enumerate(cols)))
    for r in rows:
        print(' | '.join(f'{str(r[c]):>24}' if i == 0 else f'{r[c]:>18.3f}' if isinstance(r[c], float) else f'{str(r[c]):>18}' for i, c in enumerate(cols)))
table(rows, ['arm', 'seed', 'violation', 'lifted_iou_raw', 'lifted_iou_clipped', 'chip_iou'])

/home/hasan-spark/workspace/projects/git_data/segmentation_tools_pt_sanitized/src/segmentation_tools_pt/train.py:529: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:822.)
  state.loss_dict = {k: float(v) for k, v in ld.items()}


                     arm |               seed |          violation |     lifted_iou_raw | lifted_iou_clipped |           chip_iou
             A: weight 0 |                  0 |              0.000 |              0.079 |              0.079 |              0.397
           B: weight 0.1 |                  0 |              0.000 |              0.085 |              0.085 |              0.395
           B: weight 0.5 |                  0 |              0.000 |              0.082 |              0.082 |              0.379
B: 0 → 0.5 after epoch 8 |                  0 |              0.000 |              0.082 |              0.082 |              0.403
             A: weight 0 |                  1 |              0.223 |              0.134 |              0.172 |              0.290
           B: weight 0.1 |                  1 |              0.083 |              0.161 |              0.175 |              0.302
           B: weight 0.5 |                  1 |              0.028 |              0.113 | 

In [8]:
metrics = ['violation', 'lifted_iou_raw', 'lifted_iou_clipped', 'chip_iou']
summary = []
for name in arms:
    sel = [r for r in rows if r['arm'] == name]
    summary.append({'arm': name, **{f'{m}': f"{np.mean([r[m] for r in sel]):.3f} ± {np.std([r[m] for r in sel]):.3f}" for m in metrics}})
table(summary, ['arm'] + metrics)

                     arm |          violation |     lifted_iou_raw | lifted_iou_clipped |           chip_iou
             A: weight 0 |      0.144 ± 0.102 |      0.084 ± 0.038 |      0.101 ± 0.052 |      0.349 ± 0.044
           B: weight 0.1 |      0.029 ± 0.039 |      0.105 ± 0.040 |      0.110 ± 0.046 |      0.346 ± 0.038
           B: weight 0.5 |      0.013 ± 0.012 |      0.083 ± 0.024 |      0.084 ± 0.025 |      0.315 ± 0.045
B: 0 → 0.5 after epoch 8 |      0.007 ± 0.007 |      0.098 ± 0.050 |      0.099 ± 0.051 |      0.318 ± 0.061


## 3. Reading it honestly

* **Logged term.** `logged_term` is the `train_loss_hierarchy` column that now appears in the epoch log / `metrics.csv` — it is `None` for weight `0`, so the default log is unchanged.
* **Violation (the point of the feature).** Compare `violation` of arm A with the B arms on the same seed. It should drop; where A already has ~0, there is nothing to fix.
* **Accuracy cost.** `chip_iou` and `lifted_iou_raw` should not fall by more than the seed-to-seed spread (`std` in the summary). A *large* weight switched on from epoch 0 can pull the small head down before it has learned anything — hence the warm-up arm (`w_hierarchy: 0` for the first phase). Prefer a small weight or a warm-up on real runs.
* **Clip delta.** `lifted_iou_clipped − lifted_iou_raw` is what `apply_hierarchy` buys at inference; with the loss on, the model respects the constraint already, so that gap shrinks.
* **This is a toy.** Three seeds on a tiny CNN can only show the *machinery* works. Acceptance on real data needs the A/B below, with confidence intervals from map #28 when available.

In [9]:
# soft sanity check, not a proof: on average the loss should not make the leak worse
mean_v = {n: np.mean([r['violation'] for r in rows if r['arm'] == n]) for n in arms}
print({n: round(float(v), 4) for n, v in mean_v.items()})
assert mean_v['B: weight 0.1'] <= mean_v['A: weight 0'] + 1e-9, 'hierarchy loss increased the leak on average'
assert all(r['logged_term'] is None for r in rows if r['arm'] == 'A: weight 0')
assert all(r['logged_term'] is not None for r in rows if r['arm'] == 'B: weight 0.1')

{'A: weight 0': 0.1445, 'B: weight 0.1': 0.0285, 'B: weight 0.5': 0.0126, 'B: 0 → 0.5 after epoch 8': 0.0072}


## 4. Same A/B on real data

```bash
segpt-train configs/train_multilabel_3head.yaml train.hierarchy.weight=0.0   # A  (baseline)
segpt-train configs/train_multilabel_3head.yaml train.hierarchy.weight=0.5   # B  (try 0.1 / 1.0 too; or a w_hierarchy: 0 first phase)
```

Same frozen split and seed for both. Then, per class and per image on the val split, on raw predictions and after `apply_hierarchy`:
violation rate (`metrics.subset_violation_rate` / `hierarchy_diagnostics`), pixel IoU/Dice/recall (`SegMetrics`), object recall and false alarms for `lifted` (`ObjectMetrics`).
Repeat with a second seed, and compare the worst-k `lifted` posters from `ErrorAnalysisCallback`: B should show fewer *lifted outside chip* blobs.